# IEEE-CIS Fraud Detection EDA

This notebook analyzes the IEEE-CIS Fraud Detection dataset, particularly focusing on class imbalance and the cardinality of relation columns for graph construction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure plotting
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
data_dir = '../data/raw'
transaction_file = os.path.join(data_dir, 'train_transaction.csv')

if os.path.exists(transaction_file):
    df = pd.read_csv(transaction_file)
    print(f"Loaded {len(df)} transactions.")
else:
    print(f"File not found: {transaction_file}. Please download from Kaggle.")
    # Create a dummy dataframe for demonstration if missing
    df = pd.DataFrame({'isFraud': np.random.choice([0, 1], size=1000, p=[0.965, 0.035]),
                       'card1': np.random.randint(1000, 5000, 1000),
                       'P_emaildomain': np.random.choice(['gmail.com', 'yahoo.com', 'hotmail.com'], 1000)})

## 1. Class Imbalance

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100

print("Class Counts:\n", fraud_counts)
print("\nClass Percentages:\n", fraud_pct)

plt.figure(figsize=(8, 5))
sns.barplot(x=fraud_counts.index, y=fraud_counts.values)
plt.title("Class Imbalance: isFraud")
plt.xlabel("isFraud (0 = Normal, 1 = Fraud)")
plt.ylabel("Count")
plt.show()

# Observation: Highly imbalanced dataset (~3.5% fraud), justifying the need for PR-AUC metric.

## 2. Shared Entity Cardinality (for Graph Construction)

In [ ]:
relation_cols = ['card1', 'card2', 'addr1', 'addr2', 'P_emaildomain']
# DeviceInfo is in identity table, which we could load similarly.

for col in relation_cols:
    if col in df.columns:
        counts = df[col].value_counts()
        print(f"\n--- {col} ---")
        print(f"Unique entities: {len(counts)}")
        print(f"Max occurrences of a single entity: {counts.max()}")
        print(f"Entities appearing >100 times: {(counts > 100).sum()}")

# Observation: We see high cardinality and huge hub nodes (some cards or email domains appear thousands of times).
# Creating edges between ALL transactions sharing a single email domain like 'gmail.com' would create a near-clique
# and cause memory explosion. The DEGREE_CAP=100 in build_graph.py is critical.